# 09. システム工学 — 練習問題

**対象技術**: 量子コンピューティング（量子計算リソースの導入方式選定）

システム工学の重み付きトレードスタディは、複数候補を複数の評価基準で比較し、基準重みを掛けた加重和でランキングする多基準意思決定の手法である。本ノートブックでは、量子計算リソースの導入方式（クラウド/オンプレ/ハイブリッド）を5つの基準で比較し、さらに重みを ±20% 変動させたモンテカルロ感度分析で首位の頑健性を評価する。

必要なライブラリを読み込む。

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

# --- 日本語フォント設定: 共通モジュール jp_font.py を読み込む ---
# フォント探索・登録・フォールバックの実装は repo 直下の jp_font.py に集約。
import os as _os, sys as _sys
_d = _os.path.abspath(_os.getcwd())
while not _os.path.exists(_os.path.join(_d, "jp_font.py")) and _d != _os.path.dirname(_d):
    _d = _os.path.dirname(_d)
_sys.path.insert(0, _d)
from jp_font import setup_japanese_font
setup_japanese_font()


候補と評価基準を定義する。図のラベル用に英数字IDも用意する。

In [ ]:
CANDIDATES = ["量子クラウド", "オンプレ量子機", "ハイブリッド構成"]
CRITERIA = ["初期コスト安", "セキュリティ", "性能", "人材少なさ", "拡張性"]

# 図のラベル用の英数字ID
CAND_IDS = ["Cloud", "OnPrem", "Hybrid"]
CRIT_IDS = ["Cost", "Security", "Perf", "Staff", "Scalability"]

候補×基準の評点行列を定義する。各列は「その基準で良いほど高得点」に向きを揃えてある。

In [ ]:
def build_score_matrix():
    """候補×基準の評点行列（1=劣る 〜 10=優れる）を返す。

    各列は『その基準で良いほど高得点』に向きを揃えてある
    （例:『初期コスト安』はコストが安いほど高得点）。
    """
    #               コスト  ｾｷｭﾘﾃｨ  性能  人材少  拡張性
    return np.array([
        [9,         2,      7,    8,     9],   # 量子クラウド
        [2,        10,      5,    2,     4],   # オンプレ量子機
        [5,         6,      6,    5,     7],   # ハイブリッド構成
    ], dtype=float)

加重和でランキングするトレードスタディ関数と、重みを揺らす感度分析関数を定義する。

In [ ]:
def trade_study(scores, weights):
    """加重和を計算し、候補スコアと順位を返す。"""
    w = weights / weights.sum()          # 重みを正規化
    totals = scores @ w                  # 各候補の加重総合スコア
    ranking = np.argsort(-totals)        # 降順の候補インデックス
    return totals, ranking


def sensitivity_analysis(scores, base_weights, n_trials, rng, jitter=0.2):
    """重みを ±jitter で揺らし、首位が入れ替わらない確率を求める。"""
    base_totals, base_rank = trade_study(scores, base_weights)
    base_winner = base_rank[0]
    stable = 0
    winner_count = np.zeros(len(CANDIDATES))
    for _ in range(n_trials):
        # 各重みを一様 ±jitter で乱す
        factor = 1.0 + rng.uniform(-jitter, jitter, len(base_weights))
        w = np.clip(base_weights * factor, 0.01, None)
        _, rank = trade_study(scores, w)
        winner_count[rank[0]] += 1
        if rank[0] == base_winner:
            stable += 1
    return base_winner, stable / n_trials, winner_count / n_trials

1つの重みプロファイルでトレードスタディと感度分析を実行・表示する関数を定義する。感度分析の結果（候補別の首位確率）も後の可視化のために返すようにする。

In [ ]:
def run_profile(name, scores, weights, rng):
    """1つの重みプロファイルでトレードスタディ + 感度分析を実行・表示。"""
    totals, ranking = trade_study(scores, weights)
    print(f"\n■ 重みプロファイル: {name}")
    weight_str = ", ".join(f"{c}={w:.0f}"
                           for c, w in zip(CRITERIA, weights))
    print(f"  重み = [{weight_str}]")
    for r, idx in enumerate(ranking, 1):
        print(f"   {r}位 {CANDIDATES[idx]:<14} 総合スコア {totals[idx]:.3f}")
    winner, stable_rate, wc = sensitivity_analysis(scores, weights,
                                                   4000, rng)
    print(f"  感度分析(重み±20%, 4000試行): "
          f"首位『{CANDIDATES[winner]}』の頑健性 {stable_rate*100:.1f}%")
    for idx in np.argsort(-wc):
        if wc[idx] > 0:
            print(f"     {CANDIDATES[idx]:<14} が首位になる確率 "
                  f"{wc[idx]*100:.1f}%")
    return totals, wc

評点行列を構築し、内容を表形式で表示する。

In [ ]:
rng = np.random.default_rng(2024)
scores = build_score_matrix()

print("=" * 64)
print("トレードスタディ : 量子計算リソースの導入方式選定")
print("=" * 64)
print(f"{'候補':<16}" + "".join(f"{c:>10}" for c in CRITERIA))
print("-" * 64)
for i, name in enumerate(CANDIDATES):
    print(f"{name:<16}" + "".join(f"{v:>10.0f}" for v in scores[i]))

2つの重みプロファイル（セキュリティ重視・コスト重視）でトレードスタディと感度分析を実行する。結果は可視化のため変数に保持する。

In [ ]:
# プロファイル1: セキュリティ重視（例: 金融機関）
w_sec = np.array([1, 6, 2, 1, 2], dtype=float)
totals_sec, wc_sec = run_profile("セキュリティ重視(金融機関)",
                                 scores, w_sec, rng)

# プロファイル2: コスト重視（例: スタートアップ）
w_cost = np.array([4, 1, 2, 3, 2], dtype=float)
totals_cost, wc_cost = run_profile("コスト重視(スタートアップ)",
                                   scores, w_cost, rng)

print()
print("[解釈] 同じ評点行列でも、重み配分(=組織の価値観)で首位が")
print("       入れ替わる。感度分析で頑健性が低いと出た場合は、")
print("       結論が重み設定に依存することを意思決定者に明示する。")

## 可視化: 加重スコアの棒グラフと重み感度分析

左図は2つの重みプロファイルでの候補別加重総合スコアをグループ棒グラフで示す。右図は感度分析の結果——重みを ±20% 揺らしたときに各候補が首位になる確率をプロファイル別グループ棒で示し、順位入替わりの起こりやすさを可視化する。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
x = np.arange(len(CANDIDATES))
width = 0.35

# --- 左: 加重総合スコア ---
ax = axes[0]
ax.bar(x - width / 2, totals_sec, width, color="crimson",
       label="Security-focused")
ax.bar(x + width / 2, totals_cost, width, color="steelblue",
       label="Cost-focused")
ax.set_xticks(x)
ax.set_xticklabels(CAND_IDS)
ax.set_ylabel("Weighted total score")
ax.set_title("Trade study: weighted scores by profile")
ax.legend()
ax.grid(alpha=0.3, axis="y")

# --- 右: 重み感度分析（首位になる確率）---
ax = axes[1]
ax.bar(x - width / 2, wc_sec * 100, width, color="crimson",
       label="Security-focused")
ax.bar(x + width / 2, wc_cost * 100, width, color="steelblue",
       label="Cost-focused")
ax.set_xticks(x)
ax.set_xticklabels(CAND_IDS)
ax.set_ylabel("Probability of being ranked 1st (%)")
ax.set_title("Weight sensitivity (+/-20%): rank-switch frequency")
ax.set_ylim(0, 105)
ax.legend()
ax.grid(alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

## 未来デザイン論文での使われ方と結論への影響

未来デザインの論文においてシステム工学は、達成すべき要求を明示し、複数の設計代替案を基準ごとに評価して最良案を選び出すために用いられる。論文の典型的な使われ方は、技術導入やシステム構成の選択肢を列挙し、トレードスタディで各案を加重評価して「案Aを推奨する」と結論づけることである。したがってこの手法が生み出す結論は、推奨設計案という型をとり、評価の透明性と再現可能性を備えた形で提示される。

結論の型は明快な推奨案だが、その推奨は重みと評価基準の選択の関数である。境界設定の面では、トレードスタディの表に並べた代替案と基準しか比較の俎上に乗らず、表に書かれなかった選択肢や考慮事項は最初から検討の外に置かれる。時間観の面では、未来は設定された目的関数を最大化するよう選び取る対象として扱われ、設計によって望ましい未来へ到達できるという前提が置かれる。価値の所在は、基準ごとの重みづけという明示的な数値に埋め込まれ、そこに利害や優先順位が集約される。

最大の限界は、明確な目的関数を前提とするがゆえに、深い不確実性や目的そのものをめぐる対立を扱えないことである。何を最適化すべきかが社会的に争われている状況では、システム工学は争いを解くのではなく、特定の重みづけを暗黙に採用することで争いを迂回してしまう。結果として、この手法に依拠した論文の結論は「案Aが最良である」と述べるが、それは与えられた重みと基準のもとでの最良にすぎない。推奨案は、その背後にある目的関数の正当性とともに吟味すべきものである。

## 発展課題

**課題A**: システム要素間の依存を表す設計構造行列（DSM）を題材に追加し、行と列を同時に並べ替えて強く結合した要素クラスタを浮かび上がらせる単純なクラスタリングを実装せよ。

**課題B**: 「初期コスト」と「セキュリティ」の2基準だけを軸にとり、どの候補にも支配されない（=パレート最適な）候補を抽出せよ。トレードオフのパレートフロントを表示する。